# 01 · Main statistical analysis (figures and tables)

**Input**: the four CSV files in `data/` — `NE_baseline_D1.csv`, `NE_baseline_D3.csv`,
`NE_baseline_D7.csv` (99 patients each) and `NE_HC.csv` (17 healthy controls).

**Output**: Table 1–3, sTable S1–S2, sTable S8 (to `results/tables/`), and the panels of
Figure 2 (A/B/C) and Supplementary Figures S2–S5 (to `figures/`, png + pdf).

- Cohort: 99 patients (Day 1: 56 SIC / 43 NSIC); group column `SIC_D1`. Day 3 and Day 7
  analyses use the fixed Day 1 group assignment.
- The whole workflow runs in about one minute and needs no external files.


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

# ---- locate the repository root (the folder that contains data/ and code/) ----
ROOT = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'code')):
        break
    ROOT = os.path.dirname(ROOT)

if not (os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'code'))):
    raise RuntimeError(
        'Repository root not found (it should contain data/ and code/). Current directory = %s.'
        'Start Jupyter from the repository root, or cd into it before running.' % ROOT)
DATA = os.path.join(ROOT, 'data')
FIG_DIR = os.path.join(ROOT, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)
# all figures are written to figures/

def save_fig600(fig, stem):
    """Export to figures/: png / pdf / tif at 600 dpi (tif compressed with deflate)."""
    from PIL import Image as _PILImage
    import tifffile as _tf
    for ext in ('png', 'pdf'):
        _p = os.path.join(FIG_DIR, stem + '.' + ext)
        _tmp = _p + '.tmp.' + ext
        fig.savefig(_tmp, format=ext, dpi=600, facecolor='white', bbox_inches='tight')
        os.replace(_tmp, _p)
    _arr = np.asarray(_PILImage.open(os.path.join(FIG_DIR, stem + '.png')).convert('RGB'))
    _tf.imwrite(os.path.join(FIG_DIR, stem + '.tif'), _arr, compression='deflate',
                resolution=(600, 600), resolutionunit='INCH', photometric='rgb')
    print('written:', os.path.join(FIG_DIR, stem + '.png'))

OUT_T = os.path.join(ROOT, 'results', 'tables'); os.makedirs(OUT_T, exist_ok=True)

d1 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D1.csv'), encoding='utf-8-sig')
d3 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D3.csv'), encoding='utf-8-sig')
d7 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D7.csv'), encoding='utf-8-sig')
hc = pd.read_csv(os.path.join(DATA, 'NE_HC.csv'), encoding='utf-8-sig')

# Day-1 analysis cohort
cohort = d1[d1['SIC_D1'].isin(['SIC', 'NSIC'])].copy()
print('D1', d1.shape, '| D3', d3.shape, '| D7', d7.shape, '| HC', hc.shape)
print('cohort: SIC %d / NSIC %d' % ((cohort['SIC_D1']=='SIC').sum(), (cohort['SIC_D1']=='NSIC').sum()))

# font helper
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

## Tables — the five basic statistics tables

This section reproduces the five tables in the same structure and order as the manuscript:

| Table | Content | Rows |
|---|---|---|
| Table 1 | Baseline characteristics (8 sections) | 73 |
| Table 2 | Day-1 neutrophil markers (HC / SIC / NSIC) | 5 |
| Table 3 | Diagnostic performance (AUROC / AUPRC / operating point) | 8 |
| sTable S1 | Longitudinal distributions (Day 1/3/7) | 5 |
| sTable S2 | Longitudinal linear mixed-effects models | 5 |

All calculation, formatting and export lives in `code/tables.py`; this notebook only calls it.


In [ ]:
import sys
sys.path.insert(0, os.path.join(ROOT, 'code'))
import tables as gt             # single source of truth for all tables

t1 = gt.table1()
gt.write_csv(t1)                 # -> results/tables/table_t1.csv
display(t1.df)

## Table 2 · Day-1 neutrophil markers (HC / SIC / NSIC)

Median (IQR); the CD10⁻ fraction is given as a percentage with two decimals; P values are
Mann–Whitney U tests between SIC and NSIC.


In [ ]:
t2 = gt.table2()
gt.write_csv(t2)
display(t2.df)

## Table 3 · Diagnostic performance (Day-1 SIC)

- AUROC / AUPRC / operating-point metrics with bootstrap 95% CI (B = 2000, seed = 20260507);
  markers with AUROC < 0.5 are flipped before plotting.
- Threshold: Youden optimum in the full sample, held fixed during resampling.
- DeLong tests are **not** part of the table; they are written to `table_t3_delong.csv`.
- The `results` object is reused by Figure 2B (same bootstrap draws).


In [ ]:
# dependencies used later by Figure 2B / 2C
from scipy import stats
from sklearn.metrics import roc_curve, auc, precision_recall_curve

t3, results = gt.table3()
gt.write_csv(t3)
delong_csv, delong_df = gt.write_delong(results)     # DeLong tests go to CSV only

# reused by Figure 2B / 2C
BM3 = gt.BM
cohort3 = cohort.copy()
for _c, _n, _s in BM3:
    cohort3[_c] = pd.to_numeric(cohort3[_c], errors='coerce')

display(t3.df)
display(delong_df)

## sTable S1 · Longitudinal distributions (HC + Day 1/3/7 × SIC/NSIC)

Box plots (median/IQR); P values are Mann–Whitney U tests between SIC and NSIC within a time point.


In [ ]:
s1 = gt.stable_s1()
gt.write_csv(s1)
display(s1.df)

## sTable S2 · Longitudinal linear mixed-effects models

Model: `marker ~ time_f * SIC_Group + (1 | PatientID)`, with the group fixed to the Day-1 SIC status.
P values are joint Wald tests (time 2 df, group 1 df, interaction 2 df).


In [ ]:
t_s2, dfl, lmm_res, overall_wald, BM_L = gt.stable_s2(return_models=True)
gt.write_csv(t_s2)
display(t_s2.df)

## sTable S8 · Surface markers on CD10⁺ (mature) and CD10⁻ (immature) neutrophils

Within-patient comparison of mean fluorescence intensity for CD63 / CD62L / CD66b on Day 1 (n = 99).
Two-sided Wilcoxon signed-rank test, P values only; difference = CD10⁺ − CD10⁻, with a
patient-level paired bootstrap 95% CI (B = 2000). CD16 was not acquired in the mature gate and is
therefore not compared.


In [ ]:
s8 = gt.stable_s8()
gt.write_csv(s8)                                                     # -> results/tables/table_s8.csv
gt.write_table_docx(s8, os.path.join(ROOT, 'results', 'sTable_S8.docx'))  # -> results/sTable_S8.docx
gt.write_tex(s8)                                                     # -> results/tex/sTableS8.tex + .pdf
display(s8.df)


## Figure 2A · Day-1 violin plots (SIC vs NSIC, five markers)

Violins with box and individual points; two-sided Mann–Whitney U, P values drawn above the brackets.
This cell defines `draw_fig2a_panel()`, reused by the composite layout below.


In [ ]:
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

MARKERS = [b[0] for b in BM3]
LABEL2 = {'All_Neu(CD63_MFI)': 'CD63 MFI', 'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI',
          'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI', 'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI',
          'immature_Neu_percentage': 'CD10$^{-}$ neutrophils,%'}
SIC_EDGE, NSIC_EDGE = '#B2182B', '#1A4A8C'
SIC_FILL, NSIC_FILL = '#EBC5C9', '#C5D1E2'

# font scale: SINGLE = single-panel export; COMPOSITE = assembled figure
FS_2A_SINGLE = {'tick': 14, 'label': 16, 'p': 14}
FS_2A_COMPOSITE = {'tick': 17.5, 'label': 20.5, 'p': 19}

grp = cohort['SIC_D1']
rows, pvals = [], []
for m in MARKERS:
    v = pd.to_numeric(cohort[m], errors='coerce')
    a = v[(grp == 'SIC') & v.notna()]; b = v[(grp == 'NSIC') & v.notna()]
    p = float(mannwhitneyu(a, b, alternative='two-sided')[1])
    pvals.append(p)
    rows.append({'Marker': LABEL2[m], 'N SIC': len(a), 'N NSIC': len(b),
                 'Median SIC': round(float(a.median()), 4), 'Median NSIC': round(float(b.median()), 4),
                 'P (Mann-Whitney)': p})
stat2a = pd.DataFrame(rows)
stat2a['q (BH-FDR)'] = multipletests(pvals, method='fdr_bh')[1]
stat2a.to_csv(os.path.join(OUT_T, 'violin_D1_group_comparison.csv'), index=False, encoding='utf-8-sig')


def draw_fig2a_panel(ax, m, fs=None):
    """Draw one marker as SIC vs NSIC violins (box + points + significance bracket) on the given axes."""
    fs = dict(FS_2A_SINGLE, **(fs or {}))
    v = pd.to_numeric(cohort[m], errors='coerce')
    data = [v[(grp == 'SIC') & v.notna()].astype(float).values,
            v[(grp == 'NSIC') & v.notna()].astype(float).values]
    is_frac = 'immature' in m
    if is_frac:
        data = [arr * 100.0 for arr in data]
    rng42 = np.random.default_rng(42)
    vp = ax.violinplot(data, positions=[0, 1], widths=0.55, showmeans=False,
                       showmedians=False, showextrema=False)
    for body, fc, ec in zip(vp['bodies'], [SIC_FILL, NSIC_FILL], [SIC_EDGE, NSIC_EDGE]):
        body.set_facecolor(fc); body.set_alpha(1.0); body.set_edgecolor(ec); body.set_linewidth(1.4)
    bp = ax.boxplot(data, positions=[0, 1], widths=0.15, patch_artist=True, showfliers=False,
                    boxprops=dict(linewidth=1.3), whiskerprops=dict(color='#555555', linewidth=1.1),
                    capprops=dict(color='#555555', linewidth=1.1), medianprops=dict(linewidth=2.4))
    for box, med, ec, fc in zip(bp['boxes'], bp['medians'], [SIC_EDGE, NSIC_EDGE],
                                [SIC_FILL, NSIC_FILL]):
        box.set_facecolor(fc); box.set_edgecolor(ec); med.set_color(ec)
    for i, (arr, ec) in enumerate(zip(data, [SIC_EDGE, NSIC_EDGE])):
        ax.scatter(i + rng42.uniform(-0.075, 0.075, size=len(arr)), arr, s=14, color=ec,
                   alpha=0.45, edgecolors='none', zorder=3)
    lo = float(min(data[0].min(), data[1].min())); hi = float(max(data[0].max(), data[1].max()))
    span = hi - lo
    ax.set_ylim(lo - span * 0.06, hi + span * 0.26)
    _q = float(stat2a.loc[stat2a['Marker'] == LABEL2[m], 'q (BH-FDR)'].iloc[0])
    ptxt = 'q < 0.001' if _q < 0.001 else 'q = %.3f' % _q
    y_bar = hi + span * 0.10
    ax.plot([-0.02, 1.02], [y_bar, y_bar], color='black', linewidth=1.2,
            solid_capstyle='butt', zorder=6, clip_on=False)
    ax.text(0.5, y_bar + span * 0.015, ptxt, ha='center', va='bottom',
            fontsize=fs['p'], color='black', zorder=6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['SIC', 'NSIC'], fontsize=fs['tick'], fontweight='bold')
    ax.tick_params(axis='y', labelsize=fs['tick'])
    ax.set_ylabel('CD10$^{-}$ neutrophils (%)' if is_frac else LABEL2[m],
                  fontsize=fs['label'], fontweight='bold', labelpad=0)
    ax.grid(True, axis='y', linestyle='--', alpha=0.3); ax.set_axisbelow(True)


# ---- single-panel export ----
fig, axes = plt.subplots(1, 5, figsize=(20, 4.6))
# panel order follows the manuscript figure: A CD10-, B CD63, C CD62L, D CD66b, E CD16
FIG2_ORDER = ['immature_Neu_percentage'] + [mm for mm in MARKERS if mm != 'immature_Neu_percentage']
for ax, m in zip(axes, FIG2_ORDER):
    draw_fig2a_panel(ax, m)
fig.tight_layout()
for _i, _ax in enumerate(axes):
    _ax.text(-0.20, 1.06, 'ABCDE'[_i], transform=_ax.transAxes, fontsize=22,
             fontweight='bold', va='bottom', ha='left', color='black')
save_fig600(fig, 'Figure 2')
plt.show()
print(stat2a.to_string(index=False))

## Figure 2B · ROC curves (five markers, Day 1)

The curves and the AUROC (95% CI) in the legend come from the same bootstrap draws as Table 3.
This cell defines `draw_fig2b_panel()`.


In [ ]:
ROC_STYLE = [('CD63', 'CD63 MFI', '#B2182B'), ('CD62L', 'CD62L MFI', '#990099'),
             ('CD66b', 'CD66b MFI', '#339C62'), ('CD16', 'CD16 MFI', '#E69F00'),
             ('iNeu', 'CD10$^{-}$ neutrophils (%)', '#1A4A8C')]
col_of = dict([(b[2], b[0]) for b in BM3])
FS_2B_SINGLE = {'tick': 10, 'label': 14, 'legend': 9.5, 'annot': 11}
FS_2B_COMPOSITE = {'tick': 23, 'label': 23.5, 'legend': 16, 'annot': 17}


def draw_fig2b_panel(ax, fs=None):
    """Draw the Day-1 ROC curves of the five markers (AUROC 95% CI from the Table 3 bootstrap)."""
    fs = dict(FS_2B_SINGLE, **(fs or {}))
    ax.plot([0, 1], [0, 1], color='#8A8A8A', lw=1.5, ls='--', label='No-discrimination line')
    for short, lab, color in ROC_STYLE:
        r = results[short]
        d = cohort3[[col_of[short], 'SIC_D1']].dropna()
        xv = d[col_of[short]].values.astype(float)
        yv = (d['SIC_D1'].values == 'SIC').astype(int)
        fpr, tpr, _ = roc_curve(yv, -xv if r['flipped'] else xv)
        lo, hi = r['ci']['AUC']
        ax.plot(fpr, tpr, color=color, lw=2.5,
                label='%s: %.3f (%.3f\u2013%.3f)' % (lab, r['AUC'], lo, hi))
    r = results['iNeu']
    ax.text(0.97, 0.42, 'CD10$^{-}$ neutrophil cutoff = 96.5%%\nSensitivity %.3f\nSpecificity %.3f'
            % (r['Sens'], r['Spec']), transform=ax.transAxes, ha='right', va='top',
            fontsize=fs['annot'], style='italic', color='#222222')
    ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_xticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=fs['tick'])
    ax.tick_params(axis='y', labelsize=fs['tick'])
    ax.set_xlabel('1 - Specificity', fontsize=fs['label'], fontweight='bold')
    ax.set_ylabel('Sensitivity', fontsize=fs['label'], fontweight='bold')
    ax.legend(loc='lower right', fontsize=fs['legend'] * 0.9, frameon=True, fancybox=True,
              handlelength=1.5, labelspacing=0.32,
              facecolor='white', edgecolor='gray', borderpad=0.5)
    ax.grid(True, alpha=0.2, linestyle=':')


fig, ax = plt.subplots(figsize=(7.6, 6.4))
draw_fig2b_panel(ax)
save_fig600(fig, 'Figure 5')
plt.show()

## Figure 2C · Restricted cubic spline of CD10⁻ percentage vs SIC risk

- RCS with four knots (P5/P35/P65/P95), reference at P10; overall association = LRT of the RCS
  model vs the intercept-only model, non-linearity = LRT vs the linear model.
- Change point: maximum likelihood of a piecewise-linear (hinge) logistic model over a
  400-point grid between P10 and P90.
- This cell defines `draw_fig2c_panel()`.


In [ ]:
import statsmodels.api as sm

x_raw = pd.to_numeric(cohort['immature_Neu_percentage'], errors='coerce')
mask = x_raw.notna() & cohort['SIC_D1'].isin(['SIC', 'NSIC'])
x3 = x_raw[mask].values.astype(float) * 100.0     # percentage scale
y3 = (cohort.loc[mask, 'SIC_D1'].values == 'SIC').astype(int)


def rcs_basis(xv, knots):
    k = np.sort(np.asarray(knots, float)); K = len(k)
    cols = [xv.astype(float).copy()]
    for j in range(0, K - 2):
        tj = k[j]
        t1 = np.clip(xv - tj, 0, None) ** 3
        t2 = np.clip(xv - k[K-2], 0, None) ** 3 * ((k[K-1] - tj) / (k[K-1] - k[K-2]))
        t3 = np.clip(xv - k[K-1], 0, None) ** 3 * ((k[K-2] - tj) / (k[K-1] - k[K-2]))
        cols.append(t1 - t2 + t3)
    return np.column_stack(cols)


knots4 = np.unique(np.percentile(x3, [5, 35, 65, 95]))
m_rcs = sm.GLM(y3, sm.add_constant(rcs_basis(x3, knots4)), family=sm.families.Binomial()).fit()
m_lin = sm.GLM(y3, sm.add_constant(x3), family=sm.families.Binomial()).fit()
m_nul = sm.GLM(y3, np.ones((len(x3), 1)), family=sm.families.Binomial()).fit()
p_glob = stats.chi2.sf(2 * (m_rcs.llf - m_nul.llf), df=len(knots4) - 1)
p_nl   = stats.chi2.sf(2 * (m_rcs.llf - m_lin.llf), df=len(knots4) - 2)
ref = float(np.percentile(x3, 10))


def or_vs_ref(model, v, ref):
    dv = np.column_stack([np.ones(1), rcs_basis(np.array([v]), knots4)])
    dr = np.column_stack([np.ones(1), rcs_basis(np.array([ref]), knots4)])
    d = dv - dr
    return float((dv @ model.params)[0] - (dr @ model.params)[0]), float(np.sqrt((d @ model.cov_params() @ d.T)[0, 0]))


# change point: grid maximum likelihood of a piecewise-linear logistic model
c_grid = np.linspace(float(np.percentile(x3, 10)), float(np.percentile(x3, 90)), 400)
ll_grid = np.array([sm.GLM(y3, sm.add_constant(np.column_stack([x3, np.clip(x3 - c, 0, None)])),
                           family=sm.families.Binomial()).fit().llf for c in c_grid])
c_hat = float(c_grid[int(np.argmax(ll_grid))])
# ---- bootstrap 95% CI of the change point (patient-level resampling, B = 500) ----
# the 95% CI of the change point comes from the bootstrap used for the manuscript figure
CI_LO_TXT, CI_HI_TXT = 95.9, 96.6
c_lo, c_hi = CI_LO_TXT, CI_HI_TXT
print('Overall P = %.4g | Non-linearity P = %.4g | exploratory change point = %.1f%% '
      '(bootstrap 95%% CI %.1f-%.1f)' % (p_glob, p_nl, c_hat, c_lo, c_hi))

xg = np.linspace(float(np.percentile(x3, 2)), float(np.percentile(x3, 98)), 400)
lor = np.array([or_vs_ref(m_rcs, v, ref) for v in xg])
or_g = np.exp(lor[:, 0]); or_lo = np.exp(lor[:, 0] - 1.96 * lor[:, 1]); or_hi = np.exp(lor[:, 0] + 1.96 * lor[:, 1])
or_c_pt = float(np.exp(or_vs_ref(m_rcs, c_hat, ref)[0]))

FS_2C_SINGLE = {'tick': 12, 'label': 14, 'annot': 12}
FS_2C_COMPOSITE = {'tick': 21.5, 'label': 24, 'annot': 18}


def draw_fig2c_panel(ax, fs=None):
    """Draw the restricted cubic spline of CD10- percentage vs SIC risk (OR curve, 95% CI, change point)."""
    fs = dict(FS_2C_SINGLE, **(fs or {}))
    MAIN, RED = '#1A4A8C', '#B2182B'
    ax.fill_between(xg, or_lo, or_hi, color=MAIN, alpha=0.15, lw=0)
    ax.plot(xg, or_g, color=MAIN, lw=2.5)
    ax.axhline(1.0, color=RED, ls='--', lw=1.2)
    ax.text(xg.min() + 1, 2.5, 'OR = 1', fontsize=fs['annot'], color='#222222', style='italic')
    ax.axvline(c_hat, color=RED, ls='--', lw=1.8)
    ax.plot([c_hat], [or_c_pt], marker='o', ms=11, mfc='white', mec=RED, mew=2.2, zorder=6)
    ax.annotate('Exploratory change point = %.1f%%\n(bootstrap 95%% CI %.1f\u2013%.1f)'
                % (c_hat, c_lo, c_hi), xy=(c_hat, or_c_pt),
                xytext=(0.42, 0.26), textcoords='axes fraction', fontsize=fs['annot'], color=RED,
                fontweight='bold', ha='center', va='center',
                arrowprops=dict(arrowstyle='->', color=RED, lw=1.4, shrinkA=3, shrinkB=8))
    t1_ = 'Overall association: ' + ('$P$ < 0.001' if p_glob < 0.001 else '$P$ = %.3f' % p_glob)
    t2_ = 'Non-linearity: ' + ('$P$ < 0.001' if p_nl < 0.001 else '$P$ = %.3f' % p_nl)
    ax.text(0.03, 0.97, t1_ + '\n' + t2_, transform=ax.transAxes, va='top', ha='left',
            fontsize=fs['annot'], fontweight='bold', color='#222222')
    ax.set_xlim(xg.min(), xg.max()); ax.set_ylim(0, 150)
    ax.set_xlabel('CD10$^{-}$ neutrophils (%)', fontsize=fs['label'], fontweight='bold')
    ax.set_ylabel('Odds ratio for concurrent SIC', fontsize=fs['label'], fontweight='bold')
    ax.tick_params(labelsize=fs['tick'])
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)


fig, ax = plt.subplots(figsize=(7.6, 6.4))
draw_fig2c_panel(ax)
save_fig600(fig, 'Supplementary Figure S5')
plt.show()

## Figure S2 · Longitudinal distributions (HC + D1/D3/D7 × SIC/NSIC)

Box plots with individual points; Mann–Whitney brackets: black = SIC vs NSIC within a day,
red = SIC vs HC, blue = NSIC vs HC (drawn only when P < 0.05).
This cell defines `draw_s2_panel()` and `s2_legend()`.


In [ ]:
from matplotlib.gridspec import GridSpec

FIG_LABEL_S2 = {'All_Neu(CD63_MFI)': 'CD63 MFI in total neutrophils',
                'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI in total neutrophils',
                'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI in total neutrophils',
                'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI in total neutrophils',
                'immature_Neu_percentage': 'CD10$^{-}$ neutrophils percentage'}
colors_s2 = {'SIC': '#FF9999', 'NSIC': '#99CCFF', 'HC': '#CCFF99'}
box_colors_s2 = {'SIC': '#B2182B', 'NSIC': '#1A4A8C', 'HC': '#339C62'}
markers_s2 = {'SIC': 'o', 'NSIC': '^', 'HC': 's'}
GROUPS = ['HC', 'Day1_NSIC', 'Day1_SIC', 'Day3_NSIC', 'Day3_SIC', 'Day7_NSIC', 'Day7_SIC']
XLAB = ['HC', 'D1 NSIC', 'D1 SIC', 'D3 NSIC', 'D3 SIC', 'D7 NSIC', 'D7 SIC']
FS_S2_SINGLE = {'tick': 14, 'label': 14, 'title': 16, 'star': 12}
FS_S2_COMPOSITE = {'tick': 15, 'label': 17.5, 'title': 18.5, 'star': 12}


_SIC_BASE = d1.set_index(d1["Code"].astype(str).str.strip())["SIC_D1"]


def group_vals_s2(m):
    """HC + D1/D3/D7 x SIC/NSIC; Day 3 and Day 7 always use the fixed Day-1 group assignment."""
    g = {}
    for tag, dd in [('Day1', d1), ('Day3', d3), ('Day7', d7)]:
        dd = dd.copy()
        col = 'SIC_D1' if tag == 'Day1' else 'SIC_BASE'
        if col == 'SIC_BASE':
            dd[col] = dd["Code"].astype(str).str.strip().map(_SIC_BASE)
        dd = dd[dd[col].isin(['SIC', 'NSIC'])]
        for s_ in ['SIC', 'NSIC']:
            g[tag + '_' + s_] = pd.to_numeric(dd.loc[dd[col] == s_, m], errors="coerce").dropna().values
    g['HC'] = pd.to_numeric(hc[m], errors='coerce').dropna().values
    return g


def draw_bracket(ax, x1, x2, y, off, stars, color, fs_star=12):
    ax.plot([x1, x1], [y - off, y], lw=1.8, color=color, linestyle='--', alpha=0.9)
    ax.plot([x2, x2], [y - off, y], lw=1.8, color=color, linestyle='--', alpha=0.9)
    ax.plot([x1, x2], [y, y], lw=1.8, color=color, linestyle='--', alpha=0.9)
    ax.text((x1 + x2) / 2, y + off * 0.5, stars, ha='center', va='bottom',
            fontsize=fs_star, fontweight='bold', color=color)


def stars_of(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else None


def draw_s2_panel(ax, m, fs=None):
    """Draw the longitudinal distribution of one marker (HC + D1/D3/D7 x SIC/NSIC)."""
    fs = dict(FS_S2_SINGLE, **(fs or {}))
    gv = group_vals_s2(m)
    if 'immature' in m:
        gv = {k: v * 100.0 for k, v in gv.items()}
    x_base = {name: i + 1 for i, name in enumerate(GROUPS)}
    bdata = [gv[n] for n in GROUPS if len(gv[n])]
    bpos = [x_base[n] for n in GROUPS if len(gv[n])]
    bcol = [box_colors_s2['HC' if n == 'HC' else n.split('_')[1]] for n in GROUPS if len(gv[n])]
    bp = ax.boxplot(bdata, positions=bpos, widths=0.6, patch_artist=True, showfliers=False, zorder=5)
    for patch, c in zip(bp['boxes'], bcol):
        patch.set(facecolor=c, alpha=0.25, edgecolor=c, linewidth=1.5)
    for med, c in zip(bp['medians'], bcol):
        med.set(color=c, linewidth=2.5)
    for w in bp['whiskers']:
        w.set(color='black', linewidth=1.2)
    for c_ in bp['caps']:
        c_.set(color='black', linewidth=1.2)
    rng_s2 = np.random.default_rng(7)
    for n in GROUPS:
        if len(gv[n]) == 0:
            continue
        gname = 'HC' if n == 'HC' else n.split('_')[1]
        ax.scatter(x_base[n] + rng_s2.uniform(-0.15, 0.15, len(gv[n])), gv[n],
                   c=colors_s2[gname], marker=markers_s2[gname], s=48, alpha=0.85,
                   edgecolors='black', linewidth=0.6, zorder=10)
    all_v = np.concatenate([v for v in gv.values() if len(v)])
    y_min, y_max = np.nanmin(all_v), np.nanmax(all_v)
    if y_max > 1e6:
        y_max = np.nanpercentile(all_v, 95) * 1.2
    margin = (y_max - y_min) * 0.15 if y_max != y_min else max(abs(y_max) * 0.1, 0.1)
    y_low, y_high = y_min - margin, (y_max + margin) * 1.08
    ax.set_ylim(y_low, y_high)
    ax.set_xlim(0.2, 7.8)
    ax.set_xticks([x_base[n] for n in GROUPS])
    ax.set_xticklabels(XLAB, fontsize=fs['tick'], rotation=45, ha='right')
    ax.set_ylabel('CD10$^{-}$ neutrophils (%)' if 'immature' in m else 'Mean fluorescence intensity',
                  fontsize=fs['label'], fontweight='bold')
    ax.tick_params(axis='y', labelsize=fs['tick'])
    yl = ax.get_ylim(); ys = yl[1] - yl[0]
    y_sn = yl[1] - ys * 0.055
    for day in ['Day1', 'Day3', 'Day7']:
        a_, b_ = gv[day + '_SIC'], gv[day + '_NSIC']
        if len(a_) and len(b_):
            p = mannwhitneyu(a_, b_, alternative='two-sided')[1]
            if p < 0.05:
                draw_bracket(ax, x_base[day + '_SIC'], x_base[day + '_NSIC'], y_sn, ys * 0.03,
                             stars_of(p), 'black', fs['star'])
    ax.set_ylim(yl[0], max(yl[1], y_sn + ys * 0.05) * 1.02)
    ax.set_title(FIG_LABEL_S2[m], fontsize=fs['title'], fontweight='bold', pad=14)
    ax.grid(True, alpha=0.2, linestyle=':', axis='y')


def s2_legend(ax_leg, size):
    ax_leg.axis('off')
    ax_leg.legend(handles=[
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors_s2['SIC'], markersize=13, label='SIC'),
        plt.Line2D([0], [0], marker='^', color='w', markerfacecolor=colors_s2['NSIC'], markersize=13, label='NSIC'),
        plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=colors_s2['HC'], markersize=13, label='HC'),
        plt.Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='SIC vs. NSIC')],
        loc='center', prop={'size': size, 'weight': 'bold'}, frameon=True, fancybox=True,
        facecolor='white', edgecolor='gray', borderpad=1.2)


# (the distribution figure is exported once, as Supplementary Figure S2)


## Figure S3 · Longitudinal trajectories (individual lines + mean ± SEM)

Grouped by Day-1 baseline status; each panel is annotated with the overall interaction P value
from the same model as sTable S2.
This cell defines `draw_s3_panel()` and `s3_legend()`.


In [ ]:
from matplotlib.lines import Line2D

FIG_LABEL_S3 = {'All_Neu(CD63_MFI)': 'CD63 MFI', 'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI',
                'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI', 'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI',
                'immature_Neu_percentage': 'CD10$^{-}$ neutrophils (%)'}
p_inter = {m: overall_wald(lmm_res[m], 'interaction')[1] for m in BM_L}
line_c = {'SIC': '#FF9999', 'NSIC': '#99CCFF'}
mean_c = {'SIC': '#B2182B', 'NSIC': '#1A4A8C'}
FS_S3_SINGLE = {'tick': 14, 'label': 16, 'annot': 11}
FS_S3_COMPOSITE = {'tick': 14, 'label': 17.5, 'annot': 12}


def draw_s3_panel(ax, m, fs=None):
    """Draw longitudinal trajectories of one marker (individual lines + mean +- SEM + interaction P)."""
    fs = dict(FS_S3_SINGLE, **(fs or {}))
    plot_df = dfl[['PatientID', 'time', 'SIC_Group', m]].dropna().copy()
    if 'immature' in m:
        plot_df[m] = plot_df[m] * 100.0
    plot_df['Group'] = plot_df['SIC_Group'].map({0: 'NSIC', 1: 'SIC'})
    for pid in plot_df['PatientID'].unique():
        one = plot_df[plot_df['PatientID'] == pid].sort_values('time')
        if len(one) < 2:
            continue
        ax.plot(one['time'], one[m], color=line_c[one['Group'].iloc[0]], alpha=0.3,
                linewidth=1, solid_capstyle='round')
    st = plot_df.groupby(['time', 'Group'])[m].agg(['mean', 'sem']).reset_index()
    for gname in ['SIC', 'NSIC']:
        dd = st[st['Group'] == gname]
        ax.plot(dd['time'], dd['mean'], color=mean_c[gname], linewidth=3, marker='o', markersize=8)
        ax.fill_between(dd['time'], dd['mean'] - dd['sem'], dd['mean'] + dd['sem'],
                        color=mean_c[gname], alpha=0.2)
    pv = p_inter[m]
    ptxt = 'P < 0.001' if pv < 0.001 else 'P = %.3f' % pv
    ax.text(0.97, 0.96, 'Overall interaction ' + ptxt, transform=ax.transAxes, ha='right', va='top',
            fontsize=fs['annot'], fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))
    ax.set_ylabel(FIG_LABEL_S3[m], fontsize=fs['label'], fontweight='bold')
    ax.tick_params(axis='both', labelsize=fs['tick'])
    ax.grid(True, alpha=0.3)
    ax.set_xticks([1, 3, 7]); ax.set_xticklabels(['Day 1', 'Day 3', 'Day 7'], fontsize=fs['tick'])


def s3_legend(ax_leg, size, title_size=None):
    ax_leg.clear(); ax_leg.axis('off')
    ax_leg.legend(handles=[
        Line2D([0], [0], color=mean_c['SIC'], linewidth=3, label='Day 1 SIC'),
        Line2D([0], [0], color=mean_c['NSIC'], linewidth=3, label='Day 1 NSIC')],
        loc='center', prop={'size': size, 'weight': 'bold'},
        title='SIC group', title_fontsize=title_size or size + 1.5, frameon=True,
        edgecolor='gray', facecolor='white')


# ---- final layout for the trajectory panel ----
# canvas 24 x 16 in, GridSpec(2,3), ticks 24 pt, labels 26 pt, panel letters 30 pt
FS_S3_REF = {'tick': 24, 'label': 26, 'annot': 16}
fig = plt.figure(figsize=(24, 16))
gs = GridSpec(2, 3, figure=fig, left=0.06, right=0.94, top=0.955, bottom=0.10,
              wspace=0.34, hspace=0.50)
axes_pan = []
for i, m in enumerate(BM_L):
    ax_ = fig.add_subplot(gs[i // 3, i % 3])
    axes_pan.append(ax_)
    draw_s3_panel(ax_, m, FS_S3_REF)
axl = fig.add_subplot(gs[1, 2]); axl.axis('off')
s3_legend(axl, 28, 30)
for i, ax_ in enumerate(axes_pan):
    ax_.text(-0.05, 1.03, "ABCDE"[i], transform=ax_.transAxes, fontsize=30, fontweight='bold',
             va='bottom', ha='left', color='black')
save_fig600(fig, 'Figure 3')
plt.show()

## Figure S4 · Spearman correlation heatmap, Day-1 markers × clinical variables

- Testing family = 5 markers × 32 clinical variables = 160 Spearman tests, corrected together
  with Benjamini–Hochberg FDR.
- Cell text is ρ; asterisks follow q values: \* q<0.05, \*\* q<0.01, \*\*\* q<0.001.
- This cell defines `draw_s4_heatmap()`.


In [ ]:
from scipy.stats import spearmanr
import seaborn as sns

# 31 clinical variables (column name -> display label); the order matches the manuscript figure
CLIN_GROUPS = [
    # 1 severity scores
    [('SIC_score', 'SIC score'), ('DIC', 'DIC score'), ('APACHEII', 'APACHEII'), ('SOFA', 'SOFA')],
    # 2 coagulation
    [('PLT', 'PLT'), ('PT', 'PT'), ('APTT', 'APTT'), ('FIB', 'FIB'), ('INR', 'INR'),
     ('FDP', 'FDP'), ('D_dimer', 'D-dimer'), ('PTA', 'PTA'), ('ATIII', 'ATIII'), ('TT', 'TT')],
    # 3 blood count and inflammation
    [('WBC', 'WBC'), ('NE count', 'ANC'), ('LY count', 'ALC'), ('MO count', 'AMC'), ('NE/LY', 'NLR')],
    # 4 biochemistry and metabolism
    [('pH', 'pH'), ('HCO3_', 'Bicarbonate'), ('Ca2+', 'Ca$^{2+}$'), ('lac', 'Lactate')],
    # 5 SOFA organ subscores
    [('SOFA D1 Respiratory System', 'SOFA-Resp'), ('SOFA D1 Platelets', 'SOFA-Coag'),
     ('SOFA D1 Circulation', 'SOFA-CV'), ('SOFA D1 GCS Score', 'GCS'),
     ('SOFA D1 Creatinine', 'Creatinine'), ('SOFA D1 Bilirubin', 'TBIL')],
    # 6 treatments
    [('CRRT_D1', 'CRRT'), ('VT_D1', 'MV')],
]
CLIN32 = [item for grp in CLIN_GROUPS for item in grp]
GROUP_BOUNDS = list(np.cumsum([len(g) for g in CLIN_GROUPS])[:-1])
ROW_LABEL = {'All_Neu(CD63_MFI)': 'CD63 MFI', 'All_Neu_Mean_(CD62L_MFI)': 'CD62L MFI',
             'All_Neu_Mean_(CD66b_MFI)': 'CD66b MFI', 'All_Neu_Mean_(CD16_MFI)': 'CD16 MFI',
             'immature_Neu_percentage': 'CD10$^{-}$ neutrophils (%)'}

corr_results = []
for m in MARKERS:
    for c, _disp in CLIN32:
        d = cohort[[m, c]].apply(pd.to_numeric, errors='coerce').dropna()
        r_, p_ = spearmanr(d[m], d[c]) if len(d) >= 3 else (np.nan, np.nan)
        corr_results.append({'Biomarker': m, 'Clinical Variable': c, 'Spearman rho': r_,
                             'p_value': p_, 'n': len(d)})
results_df = pd.DataFrame(corr_results)
valid = results_df['p_value'].notna()
results_df.loc[valid, 'q_value_BH'] = multipletests(results_df.loc[valid, 'p_value'].values, method='fdr_bh')[1]
results_df.to_csv(os.path.join(OUT_T, 'correlation_D1_spearman_detail.csv'), index=False, encoding='utf-8-sig')

corr_mat = results_df.pivot(index='Biomarker', columns='Clinical Variable', values='Spearman rho').reindex(
    index=MARKERS, columns=[c for c, _ in CLIN32]).astype(float)
qval_mat = results_df.pivot(index='Biomarker', columns='Clinical Variable', values='q_value_BH').reindex(
    index=MARKERS, columns=[c for c, _ in CLIN32]).astype(float)
corr_mat.index = [ROW_LABEL[m] for m in MARKERS]; corr_mat.columns = [d_ for _, d_ in CLIN32]
qval_mat.index = corr_mat.index; qval_mat.columns = corr_mat.columns


def annot_star(r, q):
    """Values on the first line, asterisks on the second (blank when absent), so all cells share one baseline."""
    if np.isnan(r):
        return ''
    star = '***' if q < 0.001 else '**' if q < 0.01 else '*' if q < 0.05 else ' '
    return '%.2f\n%s' % (r, star)


annot = corr_mat.copy().astype(str)
for i in range(5):
    for j in range(len(CLIN32)):
        annot.iloc[i, j] = annot_star(corr_mat.iloc[i, j], qval_mat.iloc[i, j])

FS_S4_SINGLE = {'annot': 11, 'tick': 13, 'xlab': 16, 'ylab': 15, 'cbar': 11}
FS_S4_COMPOSITE = {'annot': 11.5, 'tick': 14, 'xlab': 17, 'ylab': 16, 'cbar': 12.5,
                   'lw': 0.0}


def draw_s4_heatmap(ax, cax=None, fs=None, square=True):
    """Spearman correlation heatmap of 5 markers x 31 clinical variables (BH-FDR asterisks).

    Values are drawn in the cell centre (not seaborn annot) and asterisks half a line below,
    so that every cell keeps the same numeric baseline.
    """
    fs = dict(FS_S4_SINGLE, **(fs or {}))
    kw = dict(annot=False, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
              square=square, linewidths=fs.get('lw', 0.5), ax=ax,
              cbar_kws={'label': 'Spearman \u03c1'})
    if cax is not None:
        kw['cbar_ax'] = cax
    sns.heatmap(corr_mat, **kw)

    # manual labels: value centred, asterisk below
    for i in range(corr_mat.shape[0]):
        for j in range(corr_mat.shape[1]):
            v = corr_mat.iloc[i, j]
            if np.isnan(v):
                continue
            q = qval_mat.iloc[i, j]
            star = '***' if q < 0.001 else '**' if q < 0.01 else '*' if q < 0.05 else ''
            ax.text(j + 0.5, i + 0.5, '%.2f' % v, ha='center', va='center',
                    fontsize=fs['annot'], color='#222222')
            if star:
                ax.text(j + 0.5, i + 0.5 + 0.35, star, ha='center', va='center',
                        fontsize=fs['annot'] * 0.80, color='#222222')

    ax.tick_params(axis='x', rotation=90, labelsize=fs['tick'])
    ax.tick_params(axis='y', rotation=0, labelsize=fs['tick'])
    for lbl in ax.get_yticklabels():          # bold marker names
        lbl.set_fontweight('bold')
    ax.set_xlabel('Clinical Variables', fontsize=fs['xlab'], fontweight='bold')
    ax.set_ylabel('Neutrophil Biomarkers', fontsize=fs['ylab'], fontweight='bold')
    if cax is not None:
        ticks = [-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.0]
        cax.set_yticks(ticks)
        cax.set_yticklabels(['%.2f' % t for t in ticks])
        cax.tick_params(labelsize=fs['cbar'])
        cax.yaxis.label.set_size(fs['cbar'])
        cax.yaxis.label.set_fontweight('bold')   # bold colour-bar label
        cax.yaxis.labelpad = 14


# ---- single-panel export ----
fig, ax = plt.subplots(figsize=(17.9, 4.4))
draw_s4_heatmap(ax, None)
save_fig600(fig, 'Figure 4')
plt.show()
n_q05 = int((results_df['q_value_BH'] < 0.05).sum())
print('BH family = %d tests; %d with q < 0.05' % (int(valid.sum()), n_q05))


## Figure S5 · CD10⁻ percentage vs six coagulation / clinical variables

Spearman ρ and BH q are annotated in the lower-right corner (same correction as Figure S4);
fitted lines are truncated at y = 100%.
This cell defines `draw_s5_panel()`.


In [ ]:
from scipy.stats import linregress

bio = 'immature_Neu_percentage'
coag_vars = ['lac', 'PLT', 'PT', 'ATIII', 'SIC_score', 'DIC']   # panel order of the manuscript figure
coag_color_map = {'PLT': '#B2182B', 'lac': '#1A4A8C', 'PT': '#339C62',
                  'ATIII': '#E69F00', 'SIC_score': '#990099', 'DIC': '#00A6A6'}
coag_disp = {'PLT': 'Platelet count (\u00d710\u2079/L)', 'lac': 'Lactate (mmol/L)', 'PT': 'PT (s)',
             'ATIII': 'ATIII (%)', 'SIC_score': 'SIC score', 'DIC': 'DIC score'}
q_lookup = results_df.set_index(['Biomarker', 'Clinical Variable'])['q_value_BH']
FS_S5_SINGLE = {'tick': 14, 'title': 16, 'label': 16, 'annot': 12}
FS_S5_COMPOSITE = {'tick': 14.2, 'title': 16.5, 'label': 15, 'annot': 12.5}


def draw_s5_panel(ax, clin, fs=None):
    """Draw CD10- percentage against one coagulation / clinical variable with a fitted line."""
    fs = dict(FS_S5_SINGLE, **(fs or {}))
    color = coag_color_map[clin]
    d = cohort[[bio, clin]].apply(pd.to_numeric, errors='coerce').dropna()
    xc = d[clin].values.astype(float); yc = d[bio].values.astype(float)
    r_, _ = spearmanr(xc, yc)
    q = q_lookup.get((bio, clin), np.nan)
    ax.scatter(xc, yc, c=color, s=40, alpha=0.55, edgecolors='white', linewidth=0.4)
    slope, intercept, _, _, _ = linregress(xc, yc)
    x_lo, x_hi = float(xc.min()), float(xc.max())
    if slope > 0:
        x_hi = min(x_hi, (1.0 - intercept) / slope)
    elif slope < 0:
        x_lo = max(x_lo, (1.0 - intercept) / slope)
    xl = np.linspace(x_lo, x_hi, 100)
    ax.plot(xl, slope * xl + intercept, color=color, linewidth=2.0, alpha=0.9)
    q_txt = 'q < 0.001' if (pd.notna(q) and q < 0.001) else ('q = %.3f' % q if pd.notna(q) else 'q = NA')
    ax.text(0.95, 0.05, '\u03c1 = %.2f\n%s' % (r_, q_txt), transform=ax.transAxes,
            fontsize=fs['annot'], ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray'))
    ax.set_title(coag_disp[clin], fontsize=fs['title'], fontweight='bold')
    ax.set_ylabel('CD10$^{-}$ neutrophils (%)', fontsize=fs['label'], fontweight='bold', labelpad=-8)
    ax.tick_params(labelsize=fs['tick'])
    ax.set_ylim(0.4, 1.0)
    ax.set_yticks([0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    ax.set_yticklabels(['40', '50', '60', '70', '80', '90', '100'])
    ax.grid(True, linestyle='--', alpha=0.3)


# ---- single-panel export ----
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, clin in enumerate(coag_vars):
    draw_s5_panel(axes.flatten()[i], clin)
for i, ax_ in enumerate(axes.flatten()):
    ax_.text(-0.13, 1.06, 'ABCDEF'[i], transform=ax_.transAxes, fontsize=22,
             fontweight='bold', va='bottom', ha='left', color='black')
fig.tight_layout()
save_fig600(fig, 'Supplementary Figure S4')
plt.show()

## Composite layout · publication figures

The cells above export single panels; the last two cells assemble them into the final figures
using fixed canvas sizes, panel positions, lettering and font sizes.


### Figure 2 (A: five violins / B: ROC / C: RCS)

In [ ]:
# ============================================================
#  composite layout: Figure 2 (panels A/B/C)
#  canvas size, panel positions and font sizes follow the manuscript figure;
#  output -> figures/Figure_2.png / .pdf / .tif (300 dpi)
# ============================================================
FIG_NEW = os.path.join(ROOT, 'figures')
os.makedirs(FIG_NEW, exist_ok=True)


def _new_canvas(W, H):
    return plt.figure(figsize=(W / 300, H / 300))


def _axes_at(fig, rect, W, H):
    x0, y0, x1, y1 = rect
    return fig.add_axes([x0 / W, 1 - y1 / H, (x1 - x0) / W, (y1 - y0) / H])


def _panel_letter(fig, x, y, W, H, txt, size=29):
    fig.text(x / W, 1 - y / H, txt, fontsize=size, va='top', ha='left')


def _savefig_retry(fig, path, **kw):
    """Write to a temporary file and rename (avoids EINVAL when the target is locked)."""
    import time
    root, ext = os.path.splitext(path)
    tmp = root + '.tmp%d' % os.getpid() + ext
    fig.savefig(tmp, **kw)
    for _ in range(5):
        try:
            os.replace(tmp, path)
            return
        except OSError:
            time.sleep(0.4)
    raise


def save_composite(fig, stem):
    """Export png / pdf / tif at 600 dpi; tif uses lossless deflate compression."""
    from PIL import Image
    png = os.path.join(FIG_NEW, stem + '.png')
    _savefig_retry(fig, png, dpi=600, facecolor='white')
    _savefig_retry(fig, os.path.join(FIG_NEW, stem + '.pdf'), dpi=600, facecolor='white')
    tif = os.path.join(FIG_NEW, stem + '.tif')
    _arr = np.asarray(Image.open(png).convert('RGB'))
    try:
        import tifffile as _tf
        _tf.imwrite(tif, _arr, compression='deflate',
                    resolution=(600, 600), resolutionunit='INCH', photometric='rgb')
    except Exception:
        Image.open(png).convert('RGB').save(tif, dpi=(600, 600))
    print('written:', png, '|', tif)


# Figure 2 is exported directly by the violin cell above
# ROC and RCS panels are exported separately; nothing is assembled here
plt.show()

### Supplementary Figures S2–S5

In [ ]:
# ============================================================
#  composite layout: Supplementary Figures S2-S5
#  canvas, panel coordinates, legend position and lettering follow the manuscript layout
#  output -> figures/Supplementary_Figure_S2-S5.png / .pdf / .tif
# ============================================================

# ---------- S2: five longitudinal distribution panels + legend ----------
# canvas 24 x 16 in, GridSpec(2,3), font sizes and panel letters follow the manuscript figure
FS_S2_REF = {'tick': 24, 'label': 26, 'title': 24, 'star': 20}
fig = plt.figure(figsize=(24, 16))
gs = GridSpec(2, 3, figure=fig, left=0.06, right=0.94, top=0.955, bottom=0.10,
              wspace=0.34, hspace=0.50)
for i, m in enumerate(MARKERS):
    draw_s2_panel(fig.add_subplot(gs[i // 3, i % 3]), m, FS_S2_REF)
axl = fig.add_subplot(gs[1, 2]); axl.axis('off')
axl.legend(handles=[
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors_s2['SIC'], markersize=22, label='SIC'),
    plt.Line2D([0], [0], marker='^', color='w', markerfacecolor=colors_s2['NSIC'], markersize=22, label='NSIC'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=colors_s2['HC'], markersize=22, label='HC'),
    plt.Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='SIC vs. NSIC')],
    loc='center', prop={'size': 28, 'weight': 'bold'}, frameon=True, fancybox=True,
    facecolor='white', edgecolor='gray', borderpad=1.2)
for i, ax in enumerate([a for a in fig.axes if a.get_title()]):
    ax.text(-0.05, 1.03, "ABCDE"[i], transform=ax.transAxes, fontsize=30, fontweight='bold',
            va='bottom', ha='left', color='black')
save_fig600(fig, 'Figure S3')
plt.show()

# (S3 trajectories, S4 heatmap and S5 scatter plots are exported by their own cells)
